# Edge Case Handling (Tests)

In [1]:
import sys
import unittest
from pathlib import Path

import pandas as pd

BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent
sys.path.insert(0, str(BASE / "src"))

from cleaning import (clean_orders, clean_order_items,
                      check_referential_integrity, validate_emails)

def make_orders(rows):
    return pd.DataFrame(rows, columns=["order_id", "customer_id", "order_date", "status", "region_code"])


def make_items(rows):
    return pd.DataFrame(rows, columns=["item_id", "order_id", "product_id",
                                       "quantity", "unit_price", "discount_percent"])

## Defining Edge Cases

In [ ]:
class TestEdgeCases(unittest.TestCase):

    # Edge case 1: order_item referencing for an non-existent order
    def test_orphan_order_id_is_detected_and_dropped(self):
        orders = make_orders([("O1", "C1", "2026-01-01 10:00:00", "PLACED", "NORTH")])
        items = make_items([
            ("I1", "O1", "P1", 2, 10.0, 0),
            ("I2", "O999", "P1", 1, 10.0, 0),
        ])

        integrity = check_referential_integrity(items, orders)
        self.assertEqual(integrity["orphan_order_items"]["count"], 1)
        self.assertEqual(integrity["orphan_order_items"]["item_ids"], ["I2"])
        self.assertEqual(integrity["orphan_order_items"]["missing_order_ids"], ["O999"])

        cleaned, issues = clean_order_items(items, orders)
        self.assertEqual(len(cleaned), 1)
        self.assertNotIn("I2", cleaned["item_id"].values)
        self.assertEqual(issues["orphan_order_items"]["count"], 1)

    #Edge case 2: discount_percent > 100
    def test_discount_over_100_is_reported_and_clamped(self):
        orders = make_orders([("O1", "C1", "2026-01-01 10:00:00", "PLACED", "NORTH")])
        items = make_items([
            ("I1", "O1", "P1", 1, 100.0, 150),
            ("I2", "O1", "P1", 1, 100.0, 20),
        ])

        cleaned, issues = clean_order_items(items, orders)
        self.assertEqual(issues["discount_out_of_range"]["count"], 1)
        self.assertEqual(issues["discount_out_of_range"]["item_ids"], ["I1"])
        self.assertEqual(cleaned.loc[cleaned["item_id"] == "I1", "discount_percent"].iloc[0], 100)
        self.assertTrue(cleaned["discount_percent"].between(0, 100).all())
        row = cleaned[cleaned["item_id"] == "I1"].iloc[0]
        revenue = row["quantity"] * row["unit_price"] * (1 - row["discount_percent"] / 100.0)
        self.assertEqual(revenue, 0.0)

    #Edge case 3: quantity = 0
    def test_zero_quantity_is_flagged_and_contributes_no_revenue(self):
        orders = make_orders([("O1", "C1", "2026-01-01 10:00:00", "PLACED", "NORTH")])
        items = make_items([
            ("I1", "O1", "P1", 0, 50.0, 10),
            ("I2", "O1", "P1", 3, 50.0, 0),
        ])

        cleaned, issues = clean_order_items(items, orders)
        self.assertEqual(issues["zero_quantity"]["count"], 1)
        self.assertEqual(issues["zero_quantity"]["item_ids"], ["I1"])
        self.assertIn("I1", cleaned["item_id"].values)
        revenue = (cleaned["quantity"] * cleaned["unit_price"]
                   * (1 - cleaned["discount_percent"] / 100.0))
        self.assertEqual(revenue[cleaned["item_id"] == "I1"].iloc[0], 0.0)

    #Edge case 4: order_date in the future
    def test_future_order_date_is_reported_and_dropped(self):
        orders = make_orders([
            ("O1", "C1", "2026-01-01 10:00:00", "PLACED", "NORTH"),
            ("O2", "C2", "2030-01-01 10:00:00", "PLACED", "SOUTH"),
        ])

        cleaned, issues = clean_orders(orders, reference_date="2026-07-12")
        self.assertEqual(issues["future_dates"]["count"], 1)
        self.assertEqual(issues["future_dates"]["order_ids"], ["O2"])
        self.assertNotIn("O2", cleaned["order_id"].values)
        self.assertIn("O1", cleaned["order_id"].values)

    # Bonus: wrong DD-MM-YYYY date format gets fixed
    def test_wrong_date_format_is_converted(self):
        orders = make_orders([
            ("O1", "C1", "25-12-2025 09:30:00", "PLACED", "EAST"),
            ("O2", "C2", "2025-12-26 10:00:00", "PLACED", "EAST"),
        ])

        cleaned, issues = clean_orders(orders, reference_date="2026-07-12")
        self.assertEqual(issues["wrong_date_format"]["count"], 1)
        self.assertEqual(
            cleaned.loc[cleaned["order_id"] == "O1", "order_date"].iloc[0],
            "2025-12-25 09:30:00",
        )

    #Bonus: NULL / empty customer_id becomes a real missing value
    def test_null_customer_ids_are_standardized(self):
        orders = make_orders([
            ("O1", "NULL", "2026-01-01 10:00:00", "PLACED", "WEST"),
            ("O2", "", "2026-01-02 10:00:00", "PLACED", "WEST"),
            ("O3", "C3", "2026-01-03 10:00:00", "PLACED", "WEST"),
        ])

        cleaned, issues = clean_orders(orders, reference_date="2026-07-12")
        self.assertEqual(issues["null_customer_ids"]["count"], 2)
        self.assertEqual(len(cleaned), 3)
        self.assertTrue(cleaned.loc[cleaned["order_id"].isin(["O1", "O2"]), "customer_id"].isna().all())
        customers = pd.DataFrame({
            "customer_id": ["C1", "C2", "C3"],
            "email": ["good@example.com", "missing-at.example.com", "no-domain@"],
        })
        self.assertEqual(validate_emails(customers), ["C2", "C3"])

## Running all the tests

In [3]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestEdgeCases)
runner = unittest.TextTestRunner(verbosity=2, stream=sys.stdout)
result = runner.run(suite)
assert result.wasSuccessful(), "some edge-case tests failed"
print(f"\nAll {result.testsRun} edge-case tests passed.")

test_discount_over_100_is_reported_and_clamped (__main__.TestEdgeCases.test_discount_over_100_is_reported_and_clamped) ... ok
test_future_order_date_is_reported_and_dropped (__main__.TestEdgeCases.test_future_order_date_is_reported_and_dropped) ... ok
test_null_customer_ids_are_standardized (__main__.TestEdgeCases.test_null_customer_ids_are_standardized) ... ok
test_orphan_order_id_is_detected_and_dropped (__main__.TestEdgeCases.test_orphan_order_id_is_detected_and_dropped) ... ok
test_wrong_date_format_is_converted (__main__.TestEdgeCases.test_wrong_date_format_is_converted) ... ok
test_zero_quantity_is_flagged_and_contributes_no_revenue (__main__.TestEdgeCases.test_zero_quantity_is_flagged_and_contributes_no_revenue) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.137s

OK

All 6 edge-case tests passed.
